In [1]:
!pip install transformers datasets trl accelerate scikit-learn -q

In [ ]:
# Imports and Setup

import numpy as np
import pandas as pd
import os

import torch
from torch import nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from sklearn.metrics import confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoConfig,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Initialize Model and Tokenizer

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4417.19it/s]
GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
# Tokenizer Testing

text = "Hello, this is the first step of RLHF training."
tokens = tokenizer(text)
print(tokens)
print(tokenizer.decode(tokens["input_ids"]))

texts = [
    "Hello, this is the first step of RLHF training.",
    "I have a dog",
    "I also have a cat",
]

tokens_obj = tokenizer(texts)

for tokens in tokens_obj["input_ids"]:
    print(tokenizer.decode(tokens))


{'input_ids': [15496, 11, 428, 318, 262, 717, 2239, 286, 45715, 29567, 3047, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Hello, this is the first step of RLHF training.
Hello, this is the first step of RLHF training.
I have a dog
I also have a cat


In [7]:
# Load Dataset

dataset = load_dataset("sst2")

ds_train = dataset["train"].select(range(2000))
ds_val = dataset["validation"].select(range(500))

In [8]:
# Tokenize Dataset for SFT (Classification)

def tokenize(batch):
    outputs = tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    outputs["labels"] = batch["label"]
    return outputs

map_kwargs = {
    "batched": True,
    "batch_size": 512,
    "remove_columns": ["idx", "sentence", "label"],
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

In [9]:
# Filter Short Sequences

tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x["input_ids"]) > 5)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x["input_ids"]) > 5)

In [10]:
# Set Dataset Format

tokenized_dataset_train.set_format(type="torch")
tokenized_dataset_val.set_format(type="torch")

In [11]:
# Prepare DataLoaders

data_collator = DataCollatorWithPadding(tokenizer)

train_dataloader = DataLoader(
    tokenized_dataset_train,
    batch_size=32,
    shuffle=True,
    collate_fn=data_collator,
)

val_dataloader = DataLoader(
    tokenized_dataset_val,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator,
)

In [12]:
# SFT Training Setup

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optional: learning rate schedule (warmup + linear decay)
total_training_steps = num_epochs * len(train_dataloader)
warmup_steps = int(0.1 * total_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps
)

In [35]:
def train(epoch):
    model.train()
    epoch_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_dataloader:
        batch = batch.to(device)

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_train_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=-1)
        train_correct += (preds == batch["labels"]).sum().item()
        train_total += preds.numel()

    train_loss = epoch_train_loss / len(train_dataloader)
    train_acc = train_correct / train_total

    return train_loss, train_acc

In [13]:
# Validation Function

def validate(epoch):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in val_dataloader:
        batch = batch.to(device)
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += preds.numel()

    val_loss = total_loss / len(val_dataloader)
    val_acc = correct / total

    return val_loss, val_acc

In [ ]:
train_acc = []
val_acc = []
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss, train_acc = train(epoch)
    val_loss, val_acc = validate(epoch)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_acc.append(val_acc)

    print(
        f"epoch {epoch}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
        f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}"
    )

epoch 0: val_loss=0.5998, val_acc=0.7040
epoch 0: Train Loss=0.8488, Train Acc=0.5205, Val Loss=0.5998, Val Acc=0.7040
epoch 1: val_loss=0.3130, val_acc=0.8780
epoch 1: Train Loss=0.5310, Train Acc=0.7200, Val Loss=0.3130, Val Acc=0.8780
epoch 2: val_loss=0.2941, val_acc=0.8840
epoch 2: Train Loss=0.3612, Train Acc=0.8500, Val Loss=0.2941, Val Acc=0.8840
epoch 3: val_loss=0.4549, val_acc=0.8180
epoch 3: Train Loss=0.2708, Train Acc=0.8870, Val Loss=0.4549, Val Acc=0.8180
epoch 4: val_loss=0.3068, val_acc=0.8920
epoch 4: Train Loss=0.2219, Train Acc=0.9115, Val Loss=0.3068, Val Acc=0.8920
epoch 5: val_loss=0.3027, val_acc=0.8980
epoch 5: Train Loss=0.1884, Train Acc=0.9245, Val Loss=0.3027, Val Acc=0.8980
epoch 6: val_loss=0.4732, val_acc=0.8580
epoch 6: Train Loss=0.1329, Train Acc=0.9520, Val Loss=0.4732, Val Acc=0.8580
epoch 7: val_loss=0.3877, val_acc=0.8860
epoch 7: Train Loss=0.1060, Train Acc=0.9620, Val Loss=0.3877, Val Acc=0.8860
epoch 8: val_loss=0.4245, val_acc=0.8820
epoch 8

In [17]:
# Save SFT Model
os.makedirs("./sft_model_ppo", exist_ok=True)

torch.save(model.state_dict(), "./sft_model_ppo/pytorch_model.bin")
tokenizer.save_pretrained("./sft_model_epoch_1")
model.config.save_pretrained("./sft_model_epoch_1")

In [ ]:
# Load tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained("./sft_model_epoch_1")

# Load config
loaded_config = AutoConfig.from_pretrained("./sft_model_epoch_1")

# Load model
loaded_model = AutoModelForSequenceClassification.from_config(loaded_config)
loaded_model.load_state_dict(torch.load("./sft_model_ppo/pytorch_model.bin"))

print("Model loaded successfully!")
print(f"Config: {loaded_config}")


Model loaded successfully!
Config: GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": 50256,
  "problem_type": "single_label_classification",
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version"

In [22]:
# Reward Model Dataset Preparation

REWARD_TOKEN_ID = tokenizer.eos_token_id

dataset = load_dataset("sst2")
ds_train = dataset["train"].select(range(2000))
ds_val = dataset["validation"].select(range(500))

In [23]:
# Tokenization for Reward Model

def tokenize(batch):
    outputs = tokenizer(batch["sentence"])
    outputs["score"] = [0] * len(outputs["input_ids"])
    outputs["score_index"] = [0] * len(outputs["input_ids"])

    for i in range(len(outputs["input_ids"])):
        outputs["input_ids"][i].append(REWARD_TOKEN_ID)
        outputs["attention_mask"][i].append(1)
        outputs["score"][i] = float(batch["label"][i])
        outputs["score_index"][i] = len(outputs["input_ids"][i]) - 1

    return outputs

map_kwargs = {
    "batched": True,
    "batch_size": 512,
    "remove_columns": ["idx", "sentence", "label"],
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

Map: 100%|██████████| 500/500 [00:00<00:00, 23255.18 examples/s]


In [24]:
# Dataset Formatting

tokenized_dataset_train.set_format(type="torch")
tokenized_dataset_val.set_format(type="torch")

tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x["input_ids"]) > 6)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x["input_ids"]) > 6)

Filter: 100%|██████████| 500/500 [00:00<00:00, 37802.87 examples/s]


In [27]:
# Reward Model Definition

class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)

        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        return self.reward(hidden_states)


class GPT2RewardHead(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.llm = model
        self.reward_head = RewardHead(self.llm.config)

    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        last_hidden_state = transformer_outputs.hidden_states[-1]
        reward = self.reward_head(last_hidden_state).squeeze(-1)
        reward = torch.sigmoid(reward)

        return reward

In [29]:
# Initialize Reward Model

model = GPT2RewardHead(loaded_model.base_model)

data_collator = DataCollatorWithPadding(tokenizer)

train_dataloader = DataLoader(
    tokenized_dataset_train,
    batch_size=64,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_dataset_val,
    batch_size=64,
    shuffle=True,
    collate_fn=data_collator
)

In [30]:
# Reward Model Training Setup

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()

num_epochs = 10

model.to(device)

GPT2RewardHead(
  (llm): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (reward_head): RewardHead(
    (reward): Linear(in_features=768, out_features=1, bias=True)
  )
)

In [ ]:
def train(epoch):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_dataloader:
        inputs = batch.to(device)

        model_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
        }

        scores = model(**model_inputs)

        batch_indices = torch.arange(scores.shape[0], device=device)
        score = scores[batch_indices, inputs["score_index"]]

        target = inputs["score"]

        loss = criterion(score, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (score > 0.5).float()
        correct += (preds == target).sum().item()
        total += target.numel()

    train_loss = total_loss / len(train_dataloader)
    train_acc = correct / total

    return train_loss, train_acc

In [ ]:
def validate(epoch):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in val_dataloader:
        inputs = batch.to(device)

        model_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
        }

        with torch.no_grad():
            scores = model(**model_inputs)

            batch_indices = torch.arange(scores.shape[0], device=device)
            score = scores[batch_indices, inputs["score_index"]]

            target = inputs["score"]

            loss = criterion(score, target)

        total_loss += loss.item()

        preds = (score > 0.5).float()
        correct += (preds == target).sum().item()
        total += target.numel()

    val_loss = total_loss / len(val_dataloader)
    val_acc = correct / total

    return val_loss, val_acc

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_acc = train(epoch)
    val_loss, val_acc = validate(epoch)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"epoch {epoch}: "
        f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
        f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}"
    )

validation loss: 1.0415409207344055
validation loss: 0.5353597849607468
validation loss: 0.2991041298955679
validation loss: 0.43832477554678917
validation loss: 0.607815321534872
validation loss: 0.5812150053679943
validation loss: 0.5957029610872269
validation loss: 0.48245538026094437
validation loss: 0.5609032660722733
validation loss: 0.6281112246215343
validation loss: 0.6321596689522266


In [33]:
# Save and Load Reward Model

torch.save(model.state_dict(), "reward_model.pt")
model.load_state_dict(torch.load("reward_model.pt"))

<All keys matched successfully>

In [34]:
# Reward Model Evaluation

model.eval()

all_predictions = []
all_labels = []

for batch in val_dataloader:

    inputs = batch.to(device)

    model_inputs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
    }

    with torch.no_grad():

        scores = model(**model_inputs)

        batch_indices = torch.arange(scores.shape[0])

        score = scores[batch_indices, inputs["score_index"]]

        target = inputs["score"]

    predictions = (score > 0.5).int()

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(target.cpu().numpy())

confusion_matrix(all_labels, all_predictions)

array([[202,  30],
       [ 29, 235]])